In [3]:
import os
from langchain.chat_models import init_chat_model

from dotenv import load_dotenv

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model(model="groq:qwen/qwen3-32b")

In [9]:
for chunk in model.stream("hello there! wassup with yer today") :
    print(chunk.text,end="-XD-",flush=True)

-XD-<think>-XD-
-XD-Okay-XD-,-XD- the-XD- user-XD- said-XD-,-XD- "-XD-hello-XD- there-XD-!-XD- w-XD-ass-XD-up-XD- with-XD- yer-XD- today-XD-."-XD- First-XD- off-XD-,-XD- I-XD- need-XD- to-XD- figure-XD- out-XD- the-XD- context-XD- and-XD- the-XD- user-XD-'s-XD- intent-XD- here-XD-.-XD- The-XD- greeting-XD- is-XD- friendly-XD- and-XD- casual-XD-,-XD- mixing-XD- "-XD-hello-XD- there-XD-"-XD- with-XD- the-XD- more-XD- collo-XD-qu-XD-ial-XD- "-XD-w-XD-ass-XD-up-XD- with-XD- yer-XD- today-XD-."-XD- The-XD- use-XD- of-XD- "-XD-yer-XD-"-XD- instead-XD- of-XD- "-XD-you-XD-'re-XD-"-XD- suggests-XD- the-XD- user-XD- might-XD- be-XD- from-XD- a-XD- region-XD- with-XD- a-XD- distinct-XD- dialect-XD-,-XD- possibly-XD- the-XD- UK-XD- or-XD- Ireland-XD-,-XD- or-XD- maybe-XD- they-XD-'re-XD- just-XD- using-XD- it-XD- for-XD- stylist-XD-ic-XD- effect-XD-.

-XD-Looking-XD- at-XD- the-XD- user-XD-'s-XD- message-XD-,-XD- they-XD-'re-XD- likely-XD- testing-XD- how-XD- I-XD- respond-XD- to-XD- casual-XD- gr

## Pydantic stuff

This is a short exercise on making the model give response in a structured manner and how pydantic does field validation and provides a way to do nested structures within a given class ,so it gives response in a such a manner

In [10]:
from pydantic import BaseModel,Field


class Movie(BaseModel):
    title:str=Field(description="This is the title of the movie")
    year:int=Field(description="This is the year the movie was released")
    director:str=Field(description="This shows the director of the movie")
    ratings:float=Field(description="This is how much the rating of the movie is out of 5")

In [11]:
model_with_structure = model.with_structured_output(Movie)

model_with_structure

RunnableBinding(bound=ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x126d7af90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x126d7bcb0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'This is the title of the movie', 'type': 'string'}, 'year': {'description': 'This is the year the movie was released', 'type': 'integer'}, 'director': {'description': 'This shows the director of the movie', 'type': 'string'}, 'ratings': {'description': 'This is how much the rating of the movie is out of 5', 'type': 'numb

In [12]:
model.invoke("who is the director of Batman the movie")

AIMessage(content='<think>\nOkay, the user is asking who the director of the Batman movie is. Let me think. There have been several Batman movies over the years, so I need to figure out which one they\'re referring to.\n\nFirst, the original Batman from 1989. That was directed by Tim Burton. He did two Batman films: Batman (1989) and Batman Returns (1992). Then Christopher Nolan took over with Batman Begins in 2005, followed by The Dark Knight in 2008 and The Dark Knight Rises in 2012. There\'s also the more recent one, The Batman from 2022 directed by Matt Reeves.\n\nWait, the user just said "Batman the movie" without specifying. Maybe they\'re referring to the most recent one? Or maybe the original. Without more context, it\'s a bit tricky. But the question is phrased as "Batman the movie," which might be the original 1989 one. But I should check if there\'s a most well-known one.\n\nAlternatively, maybe they\'re thinking of the Christopher Nolan trilogy as the main Batman movies. No

In [13]:
model_with_structure.invoke("who is the director of Midsommar")

Movie(title='Midsommar', year=2019, director='Ari Aster', ratings=4.5)

In [15]:
from pydantic import BaseModel,Field

class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget: float | None = Field(description="The budget of the Movie in million USD")

In [16]:
model_with_structured_output = model.with_structured_output(MovieDetails)
model_with_structured_output

RunnableBinding(bound=ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x126d7af90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x126d7bcb0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'MovieDetails', 'description': '', 'parameters': {'properties': {'title': {'type': 'string'}, 'year': {'type': 'integer'}, 'cast': {'items': {'properties': {'name': {'type': 'string'}, 'role': {'type': 'string'}}, 'required': ['name', 'role'], 'type': 'object'}, 'type': 'array'}, 'genres': {'items': {'type': 'string'}, 'type': 'array'}, 'budget': {'anyOf': [{'type': 'number'}, {'type': 'null'}], 'descri

In [22]:
response=model_with_structured_output.invoke("tell me the details about the movie The Joker")
response

MovieDetails(title='The Joker', year=2019, cast=[Actor(name='Joaquin Phoenix', role='Arthur Fleck / Joker'), Actor(name='Robert De Niro', role='Murray Franklin'), Actor(name='Zazie Beetz', role='Sophie Dumond')], genres=['Crime', 'Drama', 'Thriller'], budget=55.0)

In [ ]:
# print(response.title)

The Joker


##  TypeDict

Dont need runtime validation here,How we get the response above in a certain class format we can now make it give a response in a dictionary format


In [24]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """Info about movies"""
    title:Annotated[str,...,"The title of the movie"]
    year:Annotated[int,...,"The year the movie was released"]
    director:Annotated[str,...,"The director of the movie"]
    ratings:Annotated[float,...,"This is the ratings of the movies out of 5"]


model_with_dict_output = model.with_structured_output(MovieDict)

model_with_dict_output.invoke("Tell me about the most recent avengers movie")

{'director': 'Anthony Russo, Joe Russo',
 'ratings': 4.2,
 'title': 'Avengers: Endgame',
 'year': 2019}

In [25]:
class Actor(TypedDict):
    name:str
    role:str

class MovieDetails(TypedDict):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget: float | None = Field(description="The budget of the Movie in million USD")

In [26]:
model_with_dict_output_nested = model.with_structured_output(MovieDetails)

model_with_dict_output_nested.invoke("tell me about the movie shawshank redemption")

{'budget': 25000000,
 'cast': [{'name': 'Tim Robbins', 'role': 'Andy Dufresne'},
  {'name': 'Morgan Freeman', 'role': "Ellis 'Red' Redding"}],
 'genres': ['Drama'],
 'title': 'Shawshank Redemption',
 'year': 1994}

In [27]:
model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 16384,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True}

## Data Classes

In [28]:
from pydantic import BaseModel,Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    name: str=Field(description="the name of the person")
    email: str= Field(description="this is the email id of the phone")
    phone: str= Field(description="This is the phone no of the person")




model  = init_chat_model(model = "groq:qwen/qwen3-32b" )

agent = create_agent(
    model=model,
    response_format=ContactInfo
)

result = agent.invoke({
    "messages":[{
        "role":"user",
        "content":"Extract contact info from person John Doe,john@example.com ,(555) 123-4567"
    }]
})

result

{'messages': [HumanMessage(content='Extract contact info from person John Doe,john@example.com ,(555) 123-4567', additional_kwargs={}, response_metadata={}, id='c229b237-f408-4387-a4a4-329ea96d439a'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, let me see. The user wants me to extract contact information from the given text. The input is "John Doe, john@example.com, (555) 123-4567". I need to parse this into the ContactInfo function parameters.\n\nFirst, the name is John Doe. That\'s straightforward. The email is john@example.com. The phone number is (555) 123-4567. I should check if the phone number is in the correct format. The function requires name, email, and phone as required fields. All three are present here. I just need to structure them into the JSON object as specified. Let me make sure there are no typos. Everything looks good. So the tool call should include all three parameters correctly.\n', 'tool_calls': [{'id': 'bscaww94j', 'function': {'argu

In [30]:
print(result['structured_response'])

name='John Doe' email='john@example.com' phone='(555) 123-4567'
